# Olist E-Commerce Analytics
## 02 — Data Cleaning & Transformation

### Objective
Prepare the raw Olist datasets for analytical use while preserving the original raw data.

### Main principles
- Never modify the raw CSV files.
- Document every transformation.
- Investigate missing values before treating them.
- Preserve business meaning during cleaning.
- Create analysis-ready datasets in the processed layer.

In [2]:
import pandas as pd
import numpy as np
import os

RAW_PATH = "../02_data/raw/"
PROCESSED_PATH = "../02_data/processed/"

In [3]:
orders = pd.read_csv(
    RAW_PATH + "olist_orders_dataset.csv"
)

In [4]:
orders.shape

(99441, 8)

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [7]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [8]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [10]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [11]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [12]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [13]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [14]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [15]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [16]:
orders["delivery_performance"] = np.where(
    orders["delivery_delay_days"].isna(),
    "Not Delivered",
    np.where(
        orders["delivery_delay_days"] > 0,
        "Late",
        "On Time"
    )
)

In [17]:
orders["delivery_performance"].value_counts()

delivery_performance
On Time          89941
Late              6535
Not Delivered     2965
Name: count, dtype: int64

In [18]:
orders.to_csv(
    PROCESSED_PATH + "orders_clean.csv",
    index=False
)

In [19]:
orders[[
    "order_status",
    "delivery_days",
    "delivery_delay_days",
    "delivery_performance"
]].head(10)

,order_status,delivery_days,delivery_delay_days,delivery_performance
0,delivered,8.0,-8.0,On Time
1,delivered,13.0,-6.0,On Time
2,delivered,9.0,-18.0,On Time
3,delivered,13.0,-13.0,On Time
4,delivered,2.0,-10.0,On Time
5,delivered,16.0,-6.0,On Time
6,invoiced,NaN,NaN,Not Delivered
7,delivered,9.0,-12.0,On Time
8,delivered,9.0,-32.0,On Time
9,delivered,18.0,-7.0,On Time


In [20]:
orders.groupby("delivery_performance")[
    "order_delivered_customer_date"
].apply(lambda x: x.notna().sum())

delivery_performance
Late              6535
Not Delivered        0
On Time          89941
Name: order_delivered_customer_date, dtype: int64

In [21]:
orders["delivery_days"].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [22]:
orders.loc[
    orders["delivery_days"] < 0,
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(20)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days


In [23]:
orders["delivery_delay_days"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

In [24]:
orders.loc[
    orders["delivery_delay_days"].notna(),
    "delivery_delay_days"
].sort_values(ascending=False).head(10)

55619    188.0
19590    181.0
11399    175.0
81401    167.0
89130    166.0
61610    165.0
68769    162.0
38509    161.0
40847    161.0
54480    159.0
Name: delivery_delay_days, dtype: float64

In [25]:
orders.loc[
    orders["delivery_delay_days"].notna(),
    "delivery_delay_days"
].sort_values().head(10)

40094   -147.0
15791   -140.0
57160   -135.0
86444   -124.0
67488   -109.0
79449    -84.0
62706    -83.0
63537    -78.0
95849    -78.0
37491    -78.0
Name: delivery_delay_days, dtype: float64

In [26]:
orders.loc[
    orders["delivery_days"].notna(),
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delivery_delay_days"
    ]
].sort_values(
    "delivery_days",
    ascending=False
).head(10)

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
19590,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.0,181.0
55619,1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.0,188.0
61610,440d0d17af552815d15a9e41abe49359,delivered,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.0,165.0
70307,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,2017-03-08 18:09:02,2017-09-19 14:33:17,2017-04-17,194.0,155.0
89130,285ab9426d6982034523a855f55a885e,delivered,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.0,166.0
38509,0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.0,161.0
11399,47b40429ed8cce3aee9199792275433f,delivered,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.0,175.0
81401,2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.0,167.0
54480,2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.0,159.0
62286,437222e3fd1b07396f1d9ba8c15fba59,delivered,2017-03-16 11:36:00,2017-09-19 16:28:58,2017-04-28,187.0,144.0


In [27]:
orders.loc[
    orders["delivery_delay_days"].notna(),
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delivery_delay_days"
    ]
].sort_values(
    "delivery_delay_days",
    ascending=False
).head(10)

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
55619,1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.0,188.0
19590,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.0,181.0
11399,47b40429ed8cce3aee9199792275433f,delivered,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.0,175.0
81401,2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.0,167.0
89130,285ab9426d6982034523a855f55a885e,delivered,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.0,166.0
61610,440d0d17af552815d15a9e41abe49359,delivered,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.0,165.0
68769,c27815f7e3dd0b926b58552628481575,delivered,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,187.0,162.0
38509,0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.0,161.0
40847,d24e8541128cea179a11a65176e0a96f,delivered,2017-06-12 13:14:11,2017-12-04 18:36:29,2017-06-26,175.0,161.0
54480,2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.0,159.0


In [35]:
orders["delivery_performance"] = np.select(
    [
        orders["order_delivered_customer_date"].isna() & 
        (orders["order_status"] == "delivered"),

        orders["order_delivered_customer_date"].isna(),

        orders["delivery_delay_days"] > 0,

        orders["delivery_delay_days"] == 0,

        orders["delivery_delay_days"] < 0
    ],
    [
        "Delivery Date Missing",
        "Not Delivered",
        "Late",
        "On Time",
        "Early"
    ],
    default="Unknown"
)

In [30]:
orders["delivery_performance"].value_counts()

delivery_performance
Early            88649
Late              6535
Not Delivered     2965
On Time           1292
Name: count, dtype: int64

In [31]:
orders["delivery_performance"].value_counts(
    normalize=True
).mul(100).round(2)

delivery_performance
Early            89.15
Late              6.57
Not Delivered     2.98
On Time           1.30
Name: proportion, dtype: float64

In [32]:
orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [33]:
orders.loc[
    orders["order_approved_at"].isna(),
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(20)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaT,NaT,NaT
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaT,NaT,NaT
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaT,NaT,NaT
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaT,NaT,NaT
3056,ddaec6fff982b13e7e048b627a11d6da,68f4ad79cc0c2ad06e19088f5c00e9fa,canceled,2016-10-04 19:41:32,NaT,NaT,NaT
3094,5290c34bd38a8a095b885f13958db1e1,92af427e290117f39d9ff908566072e0,canceled,2018-08-21 10:25:18,NaT,NaT,NaT
3684,03310aa823a66056268a3bab36e827fb,25dbbf0c477fd4ae0880aaffbb12e8b3,canceled,2018-08-07 16:33:59,NaT,NaT,NaT
4373,4c8b9947280829d0a8b7e81cc249b875,403c35c4d8813bf67b3d396b91ca1619,canceled,2018-08-09 14:54:47,NaT,NaT,NaT
4939,b13ea375fe9c728832688264638f84cf,0dc5884bc5ffba283678229f27e07ff9,canceled,2018-08-22 18:52:29,NaT,NaT,NaT


In [36]:
orders["delivery_performance"].value_counts()

delivery_performance
Early                    88649
Late                      6535
Not Delivered             2957
On Time                   1292
Delivery Date Missing        8
Name: count, dtype: int64

In [37]:
orders[
    orders["delivery_performance"] == "Delivery Date Missing"
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-19


In [38]:
orders["approval_date_missing"] = orders["order_approved_at"].isna()

In [39]:
orders["approval_date_missing"].value_counts()

approval_date_missing
False    99281
True       160
Name: count, dtype: int64

In [40]:
pd.crosstab(
    orders["approval_date_missing"],
    orders["order_status"]
)

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
approval_date_missing,,,,,,,,
False,2,484,0,96464,314,301,1107,609
True,0,141,5,14,0,0,0,0


In [41]:
(orders["delivery_days"] < 0).sum()

np.int64(0)

In [42]:
(
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
).sum()

np.int64(8)

In [43]:
orders["delivery_performance"].value_counts()

delivery_performance
Early                    88649
Late                      6535
Not Delivered             2957
On Time                   1292
Delivery Date Missing        8
Name: count, dtype: int64

In [34]:
orders[
    orders["delivery_performance"] == "Not Delivered"
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [44]:
orders["purchase_date"] = orders["order_purchase_timestamp"].dt.date
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_day"] = orders["order_purchase_timestamp"].dt.day
orders["purchase_day_of_week"] = orders["order_purchase_timestamp"].dt.day_name()

In [45]:
orders["purchase_year_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [46]:
orders["is_completed"] = (
    orders["order_status"] == "delivered"
)

In [47]:
orders[
    [
        "order_id",
        "order_status",
        "purchase_date",
        "purchase_year",
        "purchase_month",
        "purchase_day_of_week",
        "purchase_year_month",
        "delivery_days",
        "delivery_delay_days",
        "delivery_performance",
        "is_completed"
    ]
].head()

,order_id,order_status,purchase_date,purchase_year,purchase_month,purchase_day_of_week,purchase_year_month,delivery_days,delivery_delay_days,delivery_performance,is_completed
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02,2017,10,Monday,2017-10,8.0,-8.0,Early,True
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24,2018,7,Tuesday,2018-07,13.0,-6.0,Early,True
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08,2018,8,Wednesday,2018-08,9.0,-18.0,Early,True
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18,2017,11,Saturday,2017-11,13.0,-13.0,Early,True
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13,2018,2,Tuesday,2018-02,2.0,-10.0,Early,True


In [48]:
orders.shape

(99441, 19)

In [49]:
orders.to_csv(
    PROCESSED_PATH + "orders_clean.csv",
    index=False
)

In [50]:
os.path.exists(
    PROCESSED_PATH + "orders_clean.csv"
)

True